In [16]:
import os
import open3d as o3d
import cv2
import numpy as np

# Depth camera parameters:
FX_DEPTH = 5.8262448167737955e+02
FY_DEPTH = 5.8269103270988637e+02
CX_DEPTH = 3.1304475870804731e+02
CY_DEPTH = 2.3844389626620386e+02

def depth_to_pcd(depth_image):
    pcd = []
    height, width = depth_image.shape
    for i in range(height):
        for j in range(width):
            z = depth_image[i][j]
            x = (j - CX_DEPTH) * z / FX_DEPTH
            y = (i - CY_DEPTH) * z / FY_DEPTH
            pcd.append([x, y, z])
    pcd_o3d = o3d.geometry.PointCloud()  # create point cloud object
    pcd_o3d.points = o3d.utility.Vector3dVector(pcd)  # set pcd_np as the point cloud points
    
    
    return pcd_o3d

def rgbd_to_pcd(depth_path, rgb_path):
    color_raw = o3d.io.read_image(rgb_path)
    depth_raw = o3d.io.read_image(depth_path)
    rgbd_image = o3d.geometry.RGBDImage.create_from_color_and_depth(color_raw, depth_raw, convert_rgb_to_intensity=False)
    print(rgbd_image)
    pcd = o3d.geometry.PointCloud.create_from_rgbd_image(
        rgbd_image, 
        o3d.camera.PinholeCameraIntrinsic(o3d.camera.PinholeCameraIntrinsicParameters.PrimeSenseDefault)
        )
    # Flip it, otherwise the pointcloud will be upside down
    pcd.transform([[1, 0, 0, 0], [0, -1, 0, 0], [0, 0, -1, 0], [0, 0, 0, 1]])
    return pcd
    
    




In [17]:
def test_myself():
    depth_path = "/remote-home/liuym/data/rgbd-scenes/kitchen_small/kitchen_small_1/kitchen_small_1_70_depth.png"
    out_path = "/remote-home/liuym/data/test/"
    depth_img = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)
    pcd = depth_to_pcd(depth_img)
    #save pcd
    o3d.io.write_point_cloud(os.path.join(out_path, "kitchen_small_1_70.ply"), pcd)
    
def test_o3d():
    depth_path = "/remote-home/liuym/data/rgbd-scenes/kitchen_small/kitchen_small_1/kitchen_small_1_70_depth.png"
    rgb_path = "/remote-home/liuym/data/rgbd-scenes/kitchen_small/kitchen_small_1/kitchen_small_1_70.png"
    out_path = "/remote-home/liuym/data/test/"
    
    pcd = rgbd_to_pcd(depth_path, rgb_path)
    # colors = np.asarray(pcd.colors)
    # pcd.colors = o3d.utility.Vector3dVector(colors.astype(float) / 255.0)
    o3d.io.write_point_cloud(os.path.join(out_path, "kitchen_small_1_70_colored.ply"), pcd, write_ascii=True)

In [18]:
test_o3d()

RGBDImage of size 
Color image : 640x480, with 3 channels.
Depth image : 640x480, with 1 channels.
Use numpy.asarray to access buffer data.


In [4]:
test_myself()

In [54]:
import open3d as o3d
import numpy as np


def load_rgbd(data):
    color = o3d.io.read_image(data['color_path'])
    depth = o3d.io.read_image(data['depth_path'])
    rgbd_image = o3d.geometry.RGBDImage.create_from_color_and_depth(color, depth, convert_rgb_to_intensity=False)
    return rgbd_image

def rgbd_odometry(source_rgbd_image, target_rgbd_image, odo_init=None):
    pinhole_camera_intrinsic = o3d.camera.PinholeCameraIntrinsic(
        o3d.camera.PinholeCameraIntrinsicParameters.PrimeSenseDefault)
    option = o3d.pipelines.odometry.OdometryOption()
    if odo_init is None:
        odo_init = np.identity(4)
    print(option)
    
    success_color_term, trans_color_term, info = o3d.pipelines.odometry.compute_rgbd_odometry(
        source_rgbd_image, target_rgbd_image,
        pinhole_camera_intrinsic, odo_init,
        o3d.pipelines.odometry.RGBDOdometryJacobianFromColorTerm(), option
        )
    
    if success_color_term:
        print("Using RGB-D Odometry")
        return {'transform': trans_color_term, 'info':info}
    
    success_hybrid_term, trans_hybrid_term, info = o3d.pipelines.odometry.compute_rgbd_odometry(
        source_rgbd_image, target_rgbd_image,
        pinhole_camera_intrinsic, odo_init,
        o3d.pipelines.odometry.RGBDOdometryJacobianFromHybridTerm(), option
        )
    
    if success_hybrid_term:
        print("Using RGB-D Odometry")
        return {'transform': trans_hybrid_term, 'info':info}
    
    return None

In [55]:
import os

data_dir = "/remote-home/liuym/data/rgbd-scenes/kitchen_small/kitchen_small_1"
rgbd_file_list = ["kitchen_small_1_73", "kitchen_small_1_74", "kitchen_small_1_75", "kitchen_small_1_76"]
rgbd_data_list = []

for rgbd_file in rgbd_file_list:
    data={
            'color_path': os.path.join(data_dir, rgbd_file + ".png"),
            'depth_path': os.path.join(data_dir, rgbd_file + "_depth.png")
        }
    data['rgbd'] = load_rgbd(data)
    rgbd_data_list.append(data)

odo_init = None
relative_trans_list = []
# tgt = rgbd_data_list[-1]['rgbd']
camera_poses = []


for i in range(len(rgbd_data_list)-1):
    j = i+1
    src = rgbd_data_list[i]['rgbd']
    tgt = rgbd_data_list[j]['rgbd']
    
    # odo_init = camera_poses[-1]
    odometry = rgbd_odometry(src, tgt, odo_init=None)
    relative_trans = odometry['transform']
    relative_trans_list.append(relative_trans)
    
relative_trans_list.append(np.identity(4))
    
    
for i in range(len(rgbd_data_list)):
    trans_to_final = np.identity(4)
    for j in range(i, len(rgbd_data_list)):
        trans_to_final = relative_trans_list[j] @ trans_to_final 
    camera_poses.append(trans_to_final)
    

    


OdometryOption class.
iteration_number_per_pyramid_level = [ 20, 10, 5, ] 
depth_diff_max = 0.030000
depth_min = 0.000000
depth_max = 4.000000
Using RGB-D Odometry
OdometryOption class.
iteration_number_per_pyramid_level = [ 20, 10, 5, ] 
depth_diff_max = 0.030000
depth_min = 0.000000
depth_max = 4.000000
Using RGB-D Odometry
OdometryOption class.
iteration_number_per_pyramid_level = [ 20, 10, 5, ] 
depth_diff_max = 0.030000
depth_min = 0.000000
depth_max = 4.000000
Using RGB-D Odometry


In [39]:
camera_poses[0].dtype

dtype('float64')

In [56]:
out_path = "/remote-home/liuym/data/test_odo/kitchen_small"
for i, pose in enumerate(camera_poses):
    current_pc = o3d.geometry.PointCloud.create_from_rgbd_image(
        rgbd_data_list[i]['rgbd'], 
        o3d.camera.PinholeCameraIntrinsic(o3d.camera.PinholeCameraIntrinsicParameters.PrimeSenseDefault))
    current_pc.transform(pose)
    print(pose)
    o3d.io.write_point_cloud(os.path.join(out_path, f"kitchen_small_{i}.ply"), current_pc, write_ascii=True)
    


[[ 0.99929548 -0.02034903  0.03153521  0.10735723]
 [ 0.02161746  0.99894899 -0.0404179   0.01532022]
 [-0.0306796   0.04107114  0.9986851  -0.04989572]
 [ 0.          0.          0.          1.        ]]
[[ 0.99996065  0.00394505 -0.00794629  0.07420059]
 [-0.0042863   0.99904878 -0.04339542  0.01542461]
 [ 0.00776753  0.04342777  0.99902637 -0.04605456]
 [ 0.          0.          0.          1.        ]]
[[ 0.99919411  0.032956   -0.02291346  0.03628821]
 [-0.03350243  0.99915314 -0.02388709  0.01606225]
 [ 0.02210683  0.0246355   0.99945204 -0.02821226]
 [ 0.          0.          0.          1.        ]]
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]


In [36]:
print(camera_poses)
len(camera_poses)

[array([[1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.]]), array([[1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.]]), array([[ 0.62038107, -0.06510505,  0.78159366, -0.78779004],
       [ 0.07702609,  0.99678871,  0.02189175,  0.01871585],
       [-0.780509  ,  0.04662188,  0.62340364, -0.27886303],
       [ 0.        ,  0.        ,  0.        ,  1.        ]]), array([[ 0.89474961,  0.32638015,  0.3047936 , -0.29534868],
       [-0.14580751,  0.85863399, -0.49141413,  0.47260816],
       [-0.42209396,  0.39525141,  0.8158511 , -0.00590943],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])]


4

In [50]:
import open3d as o3d
print("3. Applying colored point cloud registration")

for i in range(4):
    result_icp = o3d.pipelines.registration.registration_colored_icp(
        source_down, target_down, radius, current_transformation,
        o3d.pipelines.registration.TransformationEstimationForColoredICP(),
        o3d.pipelines.registration.ICPConvergenceCriteria(
            relative_fitness=1e-6, relative_rmse=1e-6, max_iteration=iter))
    current_transformation = result_icp.transformation

3. Applying colored point cloud registration


NameError: name 'source_down' is not defined